# Orbit-Inertia — Train & Evaluate

**Goal:** Train a velocity-residual GRU that corrects classical dead reckoning under GNSS blackout, then evaluate against the A0 classical baseline.

This notebook **clones the code from GitHub** (code only, no dataset) and trains on the `S4_synced.csv` you uploaded.

**Setup:** use a **Python GPU (T4)** accelerator to train fast.

In [ ]:
# Clone pipeline code from GitHub (code only). Latest master.
!git clone --depth 1 https://github.com/nikhilwankhedee/orbit-inertia.git
print("Cloned.")

## Verify the dataset mount

Your uploaded dataset should appear under `/kaggle/input/<dataset-name>`. Set `DATA_ROOT` to its path. The runner looks for `S4_synced.csv` at the root (or under `processed/`).

In [ ]:
import glob
# List what Kaggle has mounted under /kaggle/input
inputs = glob.glob('/kaggle/input/**', recursive=True)
print('\n'.join(p for p in inputs if 'S4' in p or 'csv' in p))

# EDIT this to match your dataset mount
DATA_ROOT = '/kaggle/input/sih26168'
print('\nUsing DATA_ROOT =', DATA_ROOT)

## Train + Evaluate (one shot)

Runs the full pipeline: train the GRU, then recursively evaluate on the test segments and compare against the A0 baseline. All outputs land in `orbit-inertia/outputs/ml/`.

In [ ]:
!python orbit-inertia/sih26168/src/run_all.py \
    --data-root {DATA_ROOT} \
    --epochs 200 \
    --batch-size 256 \
    --hidden-size 32 \
    --context-len 20 \
    --stride 5 \
    --seed 42

## V0.2 — Deployment-Valid Phone-Only GRU

V0.1 (recursive audit) showed the V0 16-feature checkpoint is **not** deployment-valid. V0.2 retrains from scratch on **9 phone-only features** (7 phone-IMU + `nav_speed` + `nav_heading`) and evaluates fully recursively. Run this to produce the V0.2 result:

```
python .../run_all.py --data-root {DATA_ROOT} --variant v02 \
    --epochs 200 --batch-size 256 --context-len 20 --seed 42
```

Outputs land in `orbit-inertia/outputs/ml_v02/` (`v02_report.txt`, `recursive_evaluation_report.txt`, plots).

In [ ]:
!python orbit-inertia/sih26168/src/run_all.py \
    --data-root {DATA_ROOT} \
    --variant v02 \
    --epochs 200 \
    --batch-size 256 \
    --context-len 20 \
    --stride 5 \
    --seed 42